##### last_day()

- Returns the **last date** of the **month** for a given **date column**.

In [0]:
from pyspark.sql.functions import col, to_date, current_date, last_day, datediff, sum

##### 1) Basic Example with a Date Column

In [0]:
data = [("2025-02-10",), ("2025-07-15",), ("2024-02-20",)]

df_last = spark.createDataFrame(data, ["date"])
display(df_last)

date
2025-02-10
2025-07-15
2024-02-20


In [0]:
df_last.select("date", last_day("date").alias("last_day_of_month")).display()

date,last_day_of_month
2025-02-10,2025-02-28
2025-07-15,2025-07-31
2024-02-20,2024-02-29


**Note:**
- `2024 Feb = 29 days` because it is a `leap year`.

##### 2) Using last_day() with current_date()

In [0]:
spark.range(1).select(
    current_date().alias("today"),
    last_day(current_date()).alias("last_day_of_month")
).display()

today,last_day_of_month
2026-03-08,2026-03-31


##### 3) Identifying Month-End Records

- Sometimes pipelines need only the `last day of the month` records.

In [0]:
%sql
CREATE OR REPLACE TABLE account_balance (
    account_id INT,
    customer_name STRING,
    balance DOUBLE,
    balance_date DATE
);

INSERT INTO account_balance VALUES
(101, 'Shiva', 5000.50, '2025-03-28'),
(101, 'Sathya', 5200.75, '2025-03-31'),
(102, 'Albert', 3000.00, '2025-03-15'),
(102, 'Avinash', 3500.00, '2025-03-31'),
(103, 'Dilip', 4200.20, '2025-02-27'),
(103, 'Easwar', 4500.00, '2025-02-28');

SELECT * FROM account_balance;

account_id,customer_name,balance,balance_date
101,Shiva,5000.5,2025-03-28
101,Sathya,5200.75,2025-03-31
102,Albert,3000.0,2025-03-15
102,Avinash,3500.0,2025-03-31
103,Dilip,4200.2,2025-02-27
103,Easwar,4500.0,2025-02-28


In [0]:
df_ac = spark.table("account_balance")

month_end_records = df_ac \
    .withColumn("last_day_of_month", last_day(col("balance_date")))

display(month_end_records)

account_id,customer_name,balance,balance_date,last_day_of_month
101,Shiva,5000.5,2025-03-28,2025-03-31
101,Sathya,5200.75,2025-03-31,2025-03-31
102,Albert,3000.0,2025-03-15,2025-03-31
102,Avinash,3500.0,2025-03-31,2025-03-31
103,Dilip,4200.2,2025-02-27,2025-02-28
103,Easwar,4500.0,2025-02-28,2025-02-28


In [0]:
month_end_records_fltr = month_end_records.filter(
    col("balance_date") == col("last_day_of_month")
)

display(month_end_records_fltr)

account_id,customer_name,balance,balance_date,last_day_of_month
101,Sathya,5200.75,2025-03-31,2025-03-31
102,Avinash,3500.0,2025-03-31,2025-03-31
103,Easwar,4500.0,2025-02-28,2025-02-28


- `last_day(balance_date)` returns the `last date of that month`.
- This will return only the records where `balance_date = month-end` date are selected.

      data = [(101, 'Shiva', 5000.50, '2025-03-28'),
              (101, 'Sathya', 5200.75, '2025-03-31'),
              (102, 'Albert', 3000.00, '2025-03-15'),
              (102, 'Avinash', 3500.00, '2025-03-31'),
              (103, 'Dilip', 4200.20, '2025-02-27'),
              (103, 'Easwar', 4500.00, '2025-02-28')
              ]

      df = spark.createDataFrame(data, ["account_id","customer_name","balance","balance_date"])

      df.write.mode("overwrite").saveAsTable("account_balance")

##### 4) Partitioning Data by Month-End
- **Large datasets** are stored using **partitioning** for faster queries.

In [0]:
%sql
CREATE OR REPLACE TABLE sales_transactions_partitioned (
    transaction_id INT,
    customer_id INT,
    product STRING,
    transaction_date DATE,
    amount DOUBLE
);

INSERT INTO sales_transactions_partitioned VALUES
(1,101,'Laptop','2025-01-05',1200.50),
(2,102,'Phone','2025-01-15',850.00),
(3,103,'Tablet','2025-01-25',640.75),
(4,101,'Laptop','2025-02-10',900.00),
(5,104,'Phone','2025-02-18',1500.25),
(6,102,'Tablet','2025-02-28',720.00),
(7,103,'Laptop','2025-03-05',1300.00),
(8,105,'Phone','2025-03-21',980.50);

SELECT * FROM sales_transactions_partitioned;

transaction_id,customer_id,product,transaction_date,amount
1,101,Laptop,2025-01-05,1200.5
2,102,Phone,2025-01-15,850.0
3,103,Tablet,2025-01-25,640.75
4,101,Laptop,2025-02-10,900.0
5,104,Phone,2025-02-18,1500.25
6,102,Tablet,2025-02-28,720.0
7,103,Laptop,2025-03-05,1300.0
8,105,Phone,2025-03-21,980.5


In [0]:
sales_df_partitioned = spark.table("sales_transactions_partitioned")

df_partitioned = sales_df_partitioned.withColumn("month_end", last_day("transaction_date"))
display(df_partitioned)

df_partitioned.write \
              .mode("overwrite") \
              .option("mergeSchema", "true") \
              .partitionBy("month_end") \
              .saveAsTable("sales_monthly")

transaction_id,customer_id,product,transaction_date,amount,month_end
1,101,Laptop,2025-01-05,1200.5,2025-01-31
2,102,Phone,2025-01-15,850.0,2025-01-31
3,103,Tablet,2025-01-25,640.75,2025-01-31
4,101,Laptop,2025-02-10,900.0,2025-02-28
5,104,Phone,2025-02-18,1500.25,2025-02-28
6,102,Tablet,2025-02-28,720.0,2025-02-28
7,103,Laptop,2025-03-05,1300.0,2025-03-31
8,105,Phone,2025-03-21,980.5,2025-03-31


In [0]:
%sql
SELECT * FROM sales_monthly
WHERE month_end = '2025-03-31';

transaction_id,customer_id,transaction_date,amount,month_end,product
7,103,2025-03-05,1300.0,2025-03-31,Laptop
8,105,2025-03-21,980.5,2025-03-31,Phone


In [0]:
%sql
SELECT * FROM sales_monthly
WHERE month_end = '2025-02-28';

transaction_id,customer_id,transaction_date,amount,month_end,product
4,101,2025-02-10,900.0,2025-02-28,Laptop
5,104,2025-02-18,1500.25,2025-02-28,Phone
6,102,2025-02-28,720.0,2025-02-28,Tablet


##### 5) Monthly Aggregation in ETL Pipelines

**Use case:**
- `Finance dashboards` like **Power BI / Tableau** require **month-end aggregation**.

In [0]:
%sql
CREATE OR REPLACE TABLE sales_transactions (
    transaction_id INT,
    customer_id INT,
    transaction_date DATE,
    amount DOUBLE
);

INSERT INTO sales_transactions VALUES
(1, 101, '2025-01-05', 1200.50),
(2, 102, '2025-01-15', 850.00),
(3, 103, '2025-01-25', 640.75),
(4, 101, '2025-02-10', 900.00),
(5, 104, '2025-02-18', 1500.25),
(6, 102, '2025-02-28', 720.00),
(7, 103, '2025-03-05', 1300.00),
(8, 105, '2025-03-21', 980.50);

SELECT * FROM sales_transactions;

transaction_id,customer_id,transaction_date,amount
1,101,2025-01-05,1200.5
2,102,2025-01-15,850.0
3,103,2025-01-25,640.75
4,101,2025-02-10,900.0
5,104,2025-02-18,1500.25
6,102,2025-02-28,720.0
7,103,2025-03-05,1300.0
8,105,2025-03-21,980.5


In [0]:
sales_df = spark.table("sales_transactions")

monthly_sales_last = sales_df \
  .withColumn("month_end", last_day("transaction_date")).display()

transaction_id,customer_id,transaction_date,amount,month_end
1,101,2025-01-05,1200.5,2025-01-31
2,102,2025-01-15,850.0,2025-01-31
3,103,2025-01-25,640.75,2025-01-31
4,101,2025-02-10,900.0,2025-02-28
5,104,2025-02-18,1500.25,2025-02-28
6,102,2025-02-28,720.0,2025-02-28
7,103,2025-03-05,1300.0,2025-03-31
8,105,2025-03-21,980.5,2025-03-31


In [0]:
monthly_sales = sales_df \
  .groupBy(last_day("transaction_date").alias("month_end")) \
  .agg(sum("amount").alias("total_sales"))

display(monthly_sales)

month_end,total_sales
2025-01-31,2691.25
2025-02-28,3120.25
2025-03-31,2280.5


      data = [(1,101,'2025-01-05',1200.50),
              (2,102,'2025-01-15',850.00),
              (3,103,'2025-01-25',640.75),
              (4,101,'2025-02-10',900.00),
              (5,104,'2025-02-18',1500.25),
              (6,102,'2025-02-28',720.00),
              (7,103,'2025-03-05',1300.00),
              (8,105,'2025-03-21',980.50)
              ]

      sales_df = spark.createDataFrame(data, ["transaction_id","customer_id","transaction_date","amount"])

      sales_df.write.mode("overwrite").saveAsTable("sales_transactions")